In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

df_censo = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Data_Table_Relationships/src/Datas/censo_filtrado.csv", sep=";")
df_evasao = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Data_Cleaning/src/Datas/Maiores_Taxas_Evasao_e_Reprovacao_2024.csv")

df_evasao_temp = df_evasao.copy()
df_evasao_temp['evasao_medio_total'] = pd.to_numeric(df_evasao_temp['evasao_medio_total'].replace('Não informado', np.nan), errors='coerce')

df_completo = pd.merge(df_censo, df_evasao_temp, left_on='CO_ENTIDADE', right_on='codigo_escola', how='inner')
df_completo = df_completo.dropna(subset=['evasao_medio_total'])

colunas_infra = [
    'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_ESGOTO_REDE_PUBLICA',
    'IN_BANHEIRO', 'IN_COZINHA', 'IN_REFEITORIO', 'IN_QUADRA_ESPORTES',
    'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA', 'IN_SALA_DIRETORIA',
    'IN_SECRETARIA', 'IN_SALA_MULTIUSO', 'IN_SALA_LEITURA', 'IN_ALIMENTACAO',
    'IN_INTERNET', 'IN_INTERNET_ALUNOS'
]

df_completo['MEDIA_INFRAESTRUTURA'] = df_completo[colunas_infra].mean(axis=1) * 100

# Mediana
mediana_infra = df_completo['MEDIA_INFRAESTRUTURA'].median()
mediana_evasao = df_completo['evasao_medio_total'].median()


# Plotando os pontos (escolas)
fig = px.scatter(
    df_completo,
    x='MEDIA_INFRAESTRUTURA',
    y='evasao_medio_total',
    color_discrete_sequence=['#F37C20'],
    opacity=0.7,
    hover_name='CO_ENTIDADE',
    hover_data={'MEDIA_INFRAESTRUTURA': ':.1f', 'evasao_medio_total': ':.1f', 'CO_ENTIDADE': False} # Dados no hover
)

# Desenhando as linhas da Mediana (Dividindo em 4 quadrantes principais)
# Linha de Infraestrutura
fig.add_shape(
    type="line", x0=mediana_infra, y0=0, x1=mediana_infra, y1=100,
    line=dict(color="#62C4DA", width=2, dash="solid"),
    name=f'Mediana Infra ({mediana_infra:.1f}%)'
)
fig.add_annotation(
    x=mediana_infra, y=100, text=f'Mediana Infra ({mediana_infra:.1f}%)',
    showarrow=False, yshift=10, xshift=0, font=dict(color="#3498db", size=10),
    bgcolor="rgba(255,255,255,0.7)", bordercolor="#3498db", borderwidth=0.5, borderpad=2
)

fig.add_shape(
    type="line", x0=0, y0=mediana_evasao, x1=100, y1=mediana_evasao,
    line=dict(color="#7E0406", width=2, dash="solid"),
    name=f'Mediana Evasão ({mediana_evasao:.1f}%)'
)
fig.add_annotation(
    x=100, y=mediana_evasao, text=f'Mediana Evasão ({mediana_evasao:.1f}%)',
    showarrow=False, yshift=10, xshift=0, font=dict(color="#e74c3c", size=10),
    bgcolor="rgba(255,255,255,0.7)", bordercolor="#e74c3c", borderwidth=0.5, borderpad=2
)

fig.add_annotation(
    x=90, y=90, # Posicionamento no quadrante superior direito
    text="EXCEÇÃO 1:<br>Ótima Estrutura, mas<br>Alta Evasão",
    showarrow=False, font=dict(size=11, color='black', family='Arial', weight='bold'), # Cor do texto para preto
    bgcolor='rgba(255,255,255,0.8)', bordercolor='black', borderwidth=1, borderpad=4,   # Cor da borda para preto
    align="left"
)

fig.add_annotation(
    x=10, y=10, # Posicionamento no quadrante inferior esquerdo
    text="EXCEÇÃO 2:<br>Péssima Estrutura, mas<br>Retém Alunos",
    showarrow=False, font=dict(size=11, color='black', family='Arial', weight='bold'), # Cor do texto para preto
    bgcolor='rgba(255,255,255,0.8)', bordercolor='black', borderwidth=1, borderpad=4,   # Cor da borda para preto
    align="left"
)

cor_fundo = '#F8F9FA'

fig.update_layout(
    title=dict(text='<b>A RELAÇÃO CONTROVERSA: ANOMALIAS NA EVASÃO ESCOLAR BRASILEIRA</b>', x=0.5, font=dict(size=16, color='#111111')),
    xaxis_title='Índice de Infraestrutura Escolar (%)',
    yaxis_title='Taxa de Evasão Escolar (%)',
    xaxis_range=[0, 100],
    yaxis_range=[0, 100],

    plot_bgcolor=cor_fundo,
    paper_bgcolor=cor_fundo,

    hovermode="closest",
    showlegend=False,
    margin=dict(l=40, r=40, t=80, b=40)
)

fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(221, 127, 159, 0.4)', griddash='dash')
fig.update_yaxes(showgrid=True, gridwidth=0.2, gridcolor='rgba(221, 127, 159, 0.4)', griddash='dash')

fig.show()